## 003 - AppleCider Metadata
<a id='index'></a> <br>

- [part 1: import stuff, other basic things](#import)
    - if you haven't already processed the dataset into alerts, or created `data_train.csv`, `test_df.csv`, etc, go back to the previous notebook [001: data pre-processing walkthrough](https://github.com/ajunell/AppleCider/blob/main/notebooks/001-data-processing.ipynb). this won't work unless you've done all the preprocessing steps! 
- [part 2: Dataset](#dataset)

In [1]:
import pandas as pd
import numpy as np
import os
from tqdm.auto import tqdm
import random
import torch
import joblib
from torch import nn
import sys
sys.path.insert(0, 'AppleCider')


from AppleCider.core.dataset import DataGenerator
import AppleCider.core.drink as drink
from datetime import datetime

from math import sqrt
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from scipy.interpolate import interp1d 
from scipy import stats
import pickle 

In [2]:
data_dir = 'SEDM_folder'
SEDM_dataset = pd.read_csv('/csv-pkl/SEDM_BrightTransientSurvey.csv')

TEST_DATA_PATH = 'data_test_BTS'
TRAIN_DATA_PATH = 'data_train_BTS'

CLASSES = ['SN Ia', 'SN II', 'SN IIP', 'Cataclysmic', 'AGN', 'SN IIn', 'SN Ic', 'SN Ib', 'SN IIb', 'Tidal Disruption Event']


id2target = {i: CLASSES[i] for i in range(10)}
target2id = {v: k for k, v in id2target.items()}

data_train_BTS = pd.read_csv('/csv-pkl/data_train_BTS.csv')

data_train_BTS['type_encoded'] = data_train_BTS['type'].map(target2id)


In [3]:
file_path= '/csv-pkl/'
filename = os.path.join(file_path, 'test_files_BTS.pkl')
filename_train = os.path.join(file_path, 'train_files_BTS.pkl')
filename_val = os.path.join(file_path, 'val_files_BTS.pkl')

## write to file using pickle
#with open(filename, 'wb') as file:
#    pickle.dump(test_files, file)

#with open(filename_train, 'wb') as file:
#    pickle.dump(train_files, file)
#
#with open(filename_val, 'wb') as file:
#    pickle.dump(val_files, file)

## load files
with open(filename, 'rb') as file:
    test_files = pickle.load(file)
with open(filename_train, 'rb') as file:
    train_files = pickle.load(file)
with open(filename_val, 'rb') as file:
    val_files = pickle.load(file)

<a id='import'></a><br>


<big>part 2: Dataset</big><br>


<a id='dataset'></a><br>
<i><small>[back to index](#index)</small></i>

In [5]:
from torch import nn

class DataGenerator_notebook(torch.utils.data.Dataset):

    def __init__(self, preprocessed_path, df, step, file_list=None, **kwargs):
        super().__init__(**kwargs)
        self.preprocessed_path = preprocessed_path
        self.step = step
        self.df = df
        
        
        self.id2target = {i: x for i, x in enumerate(sorted(self.df[self.step].unique()))}
        self.target2id = {'SN Ia': 0 , 'SN Ic': 0,  'SN Ib': 0, 'SN II': 1, 'SN IIP': 1, 'SN IIn': 1, 'SN IIb': 1,
                          'Cataclysmic': 2, 'AGN': 3, 'Tidal Disruption Event': 4}

        if file_list is not None:
            self.data_files = file_list
        else:
            self.data_files = [f for f in os.listdir(preprocessed_path) if f.endswith('.npy')]
        
    def __len__(self): 
        return(len(self.data_files))
    
    def __getitem__(self, index):    
        ''' load processed object alerts to get photometry, metadata, images''' 
        file_path = os.path.join(self.preprocessed_path, str(self.data_files[index]))
        sample = np.load(file_path, allow_pickle=True).item()
        
        obj_id = sample['obj_id']
        photometry = sample['photometry']
        metadata = sample['metadata'].to_numpy()
        images = sample['images']
        spectra = sample['spectra']

        # get spectra csv, save wavelengths fluxes
        obj_id_alert = str(self.data_files[index])
        obj_id = obj_id_alert[:12] # only includes ZTFID from 'ZTFID_alerts.npy' 
        
        # get label
        obj_df = self.df[self.df['name'] == obj_id]
        obj_label = obj_df['type_encoded'].iloc[0]
        
        # convert photometry, metadata, images, spectra to tensors
        photometry_tensor = torch.tensor(photometry, dtype=torch.float32)
        #padd tensors
        photo_len = len(photometry_tensor)
        max_photo = 225  # maximum photometry length from an alert
        add_dim = max_photo - photo_len
        
        # padded photometry so all photometry the same length
        if photo_len <= 225:
            photometry_padded = nn.ConstantPad1d((0, 0, 0, add_dim), 0)(photometry_tensor)
        else:
            # check max photo length from alerts again! 
            print("too much photometry. try again!", photo_len)
            
        photometry_mask = torch.ones((photometry_padded.size(0), photometry_padded.size(1)))    
            
        #metadata_tensor = torch.tensor(metadata)
        images_tensor = torch.tensor(images)
        spectra_tensor = torch.from_numpy(spectra)
        # convert label to tensor
        target = torch.tensor(obj_label).type(torch.LongTensor)  
        
        # functionally these are blanks
        blank_spectra = torch.zeros((225, 4))
        blank_images = torch.zeros((225, 4))

        return photometry_padded, photometry_mask, metadata, blank_images, blank_spectra, target

In [6]:
def collate_func(data):
    
    photometry, metadata, images, spectra, labels = zip(*data)
    
    labels = torch.tensor(labels, dtype=torch.int64)
    
    photometry = torch.stack(photometry)
    photometry_mask = torch.ones((photometry.size(0), photometry.size(1)))
    
    metadata = torch.stack(metadata)
    images = torch.stack(images)
    spectra = torch.stack(spectra)
    
    
    return photometry, photometry_mask, metadata, images, spectra, labels

In [7]:
train_dataset = DataGenerator_notebook(TRAIN_DATA_PATH, data_train_BTS, 'type', file_list=train_files)
val_dataset = DataGenerator_notebook(TRAIN_DATA_PATH, data_train_BTS, 'type', file_list=val_files)

In [8]:
_, _,metadata, _,_,_ = train_dataset[50]
metadata.shape,\
metadata.dtype

((10,), dtype('float64'))

In [9]:
metadata = [el[2] for el in train_dataset]
metadata = np.array(metadata)

In [10]:
scaler = StandardScaler()
scaler.fit(metadata)

StandardScaler()

In [11]:
print("Column means:", scaler.mean_)
print("Column standard deviations:", np.sqrt(scaler.var_))

Column means: [-2.39539543e-01 -1.41397376e+01  3.80029309e+00 -6.19181212e+00
  1.81732694e+02  2.59561737e+01  9.06479237e+00  3.30375021e-01
  2.66218730e+01  8.30248060e-02]
Column standard deviations: [ 21.99606951 119.63766541   3.51895802 120.73624831 101.79210959
  25.39026745  23.62693778   0.26117169  16.90789201   1.87442516]


In [12]:
metadata[0]

array([ 9.58332978e-03,  9.84207988e-01,  3.82363737e-01,  2.53998528e+01,
        4.94659272e+01, -1.04955800e-01,  3.00000000e+00,  4.28000003e-01,
        9.87475204e+00,  6.30998373e-01])

In [13]:
scaler.transform(metadata[0].reshape(1,-1))[0]

array([ 0.01132579,  0.12641458, -0.97129017,  0.26165849, -1.29938133,
       -1.02642202, -0.25668973,  0.37379619, -0.99049136,  0.2923422 ])

In [14]:
def get_metadata_scaler(train_dataset, metadata_scaler_path):
    
    metadata = [el[2] for el in train_dataset]
    metadata = np.array(metadata)
    
    scaler = StandardScaler()
    scaler.fit(metadata)
    
    print("Column means:", scaler.mean_)
    print("Column standard deviations:", np.sqrt(scaler.var_),"\n")
    
    print("save to:", metadata_scaler_path)
    joblib.dump(scaler, os.path.join(metadata_scaler_path,'scaler_BTS.pkl'))


In [15]:
get_metadata_scaler(train_dataset, 'csv-pkl/')

Column means: [-2.39539543e-01 -1.41397376e+01  3.80029309e+00 -6.19181212e+00
  1.81732694e+02  2.59561737e+01  9.06479237e+00  3.30375021e-01
  2.66218730e+01  8.30248060e-02]
Column standard deviations: [ 21.99606951 119.63766541   3.51895802 120.73624831 101.79210959
  25.39026745  23.62693778   0.26117169  16.90789201   1.87442516] 

save to: /Users/junell/Documents/AppleCider/csv-pkl/
